# RQ3 Gap/Deficit Extended Model Selection

This notebook extends the model-selection evidence for two bounded outcomes: `sufficiency_gap` and `sufficiency_deficit`. It keeps the same candidate model families as the RQ3 main selection stage:

- **M1**: full social specification with age and household size.
- **M4**: reduced specification without age and household size.

The notebook runs:
1. Region-stratified fractional logit models for each outcome.
2. Global fractional logit interaction models for all requested interaction pairs.
3. Simple-slope / AME summaries for interaction interpretation.

All outputs are stored under the existing `Main_model_selection` folder so final model selection can compare all candidate branches together.

In [ ]:

from pathlib import Path
import json
import sys
import warnings

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
LOCAL_PKG = PROJECT / 'Process/python_packages'
if LOCAL_PKG.exists() and str(LOCAL_PKG) not in sys.path:
    sys.path.insert(0, str(LOCAL_PKG))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

DATA_PATH = PROJECT / 'Data/global_SE_EC_sufficiency_controls_2015.csv'
RESULTS_DIR = PROJECT / 'Results/RQ3_social_determinants/Main_model_selection'
TABLE_DIR = RESULTS_DIR / 'Tables'
FIG_DIR = RESULTS_DIR / 'Figures'
INTERACTION_DIR = RESULTS_DIR / 'Interaction_candidates'
PROCESS_DIR = PROJECT / 'Process/RQ3_main_model_selection'
for d in [RESULTS_DIR, TABLE_DIR, FIG_DIR, INTERACTION_DIR, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Data:', DATA_PATH)
print('Model-selection results:', RESULTS_DIR)


In [ ]:

def clean_model_data(df):
    d = df.copy()
    d['income_clean'] = d['income'].where(d['income'] >= 0, np.nan)
    d['wealth_clean'] = d['wealth'].where(d['wealth'] >= 0, np.nan)
    d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
    d['female_share'] = d['gender']
    d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))
    d['GDP_pc'] = d['GDP'] / d['population'].replace(0, np.nan)
    d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
    d['log_population'] = np.log(d['population'].where(d['population'] > 0))
    return d

raw = pd.read_csv(DATA_PATH)
df = clean_model_data(raw)

controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
core_predictors = ['SES_raw', 'rural', 'education', 'female_share']
M1_predictors = core_predictors + ['age', 'Hhsize'] + controls
M4_predictors = core_predictors + controls
model_families = {'M1_full_age_hhsize': M1_predictors, 'M4_drop_age_hhsize': M4_predictors}
outcomes = ['sufficiency_gap', 'sufficiency_deficit']

all_needed = outcomes + ['iso_code', 'country', 'GDLcode', 'analytical_region', 'population'] + M1_predictors
sample = df.dropna(subset=all_needed).copy()
sample = sample[(sample['population'] > 0)].copy()
for y in outcomes:
    sample = sample[(sample[y] >= 0) & (sample[y] <= 1)].copy()

# Standardise on the common M1/M4 sample so comparisons are not driven by sample changes.
std = sample.copy()
standardization_rows=[]
for var in sorted(set(M1_predictors)):
    mu = std[var].mean()
    sd = std[var].std(ddof=0)
    std['z_' + var] = (std[var] - mu) / sd
    standardization_rows.append({'variable': var, 'mean': float(mu), 'sd': float(sd)})
pd.DataFrame(standardization_rows).to_csv(PROCESS_DIR / 'rq3_gap_deficit_model_selection_standardization_parameters.csv', index=False)

print('Common gap/deficit sample rows:', len(std))
print('Countries:', std['iso_code'].nunique())
print('Regions:', std['analytical_region'].nunique())
print(std.groupby('analytical_region').size())
print(std[outcomes].describe())


In [ ]:

def star(p):
    if pd.isna(p):
        return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.1: return '†'
    return ''

def fit_fractional_logit(data, outcome, rhs_terms):
    formula = outcome + ' ~ ' + ' + '.join(rhs_terms)
    return smf.glm(formula, data=data, family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': data['iso_code']}, maxiter=300
    )

def main_rhs(predictors):
    return ['z_' + v for v in predictors] + ['C(iso_code)']

def ame_factor(fit, data):
    mu = fit.predict(data)
    return float(np.mean(mu * (1 - mu)))

def extract_fractional_main_effects(fit, data, outcome, model, scope, predictors, region=None):
    rows=[]
    ci = fit.conf_int()
    factor = ame_factor(fit, data)
    for var in predictors:
        param = 'z_' + var
        if param not in fit.params.index:
            continue
        b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
        lo, hi = ci.loc[param].tolist()
        rows.append({
            'outcome': outcome, 'scope': scope, 'region': region, 'model': model, 'variable': var,
            'logit_coef_per_sd': b, 'cluster_se': se,
            'ci_low': float(lo), 'ci_high': float(hi), 'p_value': p, 'stars': star(p),
            'AME_per_sd': b * factor,
            'AME_ci_low_approx': float(lo) * factor,
            'AME_ci_high_approx': float(hi) * factor,
            'ame_factor_mean_mu_1_minus_mu': factor,
            'n_obs': int(fit.nobs), 'n_countries': data['iso_code'].nunique(),
            'aic': float(fit.aic), 'bic_llf': float(getattr(fit, 'bic_llf', np.nan))
        })
    return rows

interaction_pairs = [
    ('rural', 'SES_raw', 'Rurality x SES'),
    ('rural', 'education', 'Rurality x Education'),
    ('female_share', 'SES_raw', 'Female share x SES'),
    ('female_share', 'education', 'Female share x Education'),
    ('SES_raw', 'education', 'SES x Education'),
    ('age', 'female_share', 'Age x Female share'),
    ('SES_raw', 'Hhsize', 'SES x Household size'),
    ('female_share', 'rural', 'Female share x Rurality'),
]


In [ ]:

# Region-stratified fractional logit main-effect models for gap and deficit.
region_rows=[]
region_summary=[]
for outcome in outcomes:
    for model_name, predictors in model_families.items():
        for region, dreg in std.groupby('analytical_region', dropna=False):
            dreg = dreg.copy()
            n_countries = dreg['iso_code'].nunique()
            n_obs = len(dreg)
            if n_countries < 3 or n_obs < 30:
                region_summary.append({
                    'outcome': outcome, 'model': model_name, 'region': region, 'n_obs': n_obs,
                    'n_countries': n_countries, 'status': 'skipped_too_few_clusters'
                })
                continue
            try:
                fit = fit_fractional_logit(dreg, outcome, main_rhs(predictors))
                region_rows.extend(extract_fractional_main_effects(fit, dreg, outcome, model_name, 'region', predictors, region=region))
                region_summary.append({
                    'outcome': outcome, 'model': model_name, 'region': region,
                    'n_obs': n_obs, 'n_countries': n_countries, 'status': 'ok',
                    'aic': fit.aic, 'bic_llf': getattr(fit, 'bic_llf', np.nan)
                })
            except Exception as e:
                region_summary.append({
                    'outcome': outcome, 'model': model_name, 'region': region,
                    'n_obs': n_obs, 'n_countries': n_countries, 'status': 'failed', 'error': repr(e)
                })

region_frac_coef = pd.DataFrame(region_rows)
region_frac_summary = pd.DataFrame(region_summary)
region_frac_coef.to_csv(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_region_coefficients_M1_M4.csv', index=False)
region_frac_summary.to_csv(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_region_summary_M1_M4.csv', index=False)
region_frac_summary


In [ ]:

# Global fractional logit interaction models for gap and deficit.
interaction_rows=[]
simple_slope_rows=[]
fit_rows=[]
for outcome in outcomes:
    for model_name, base_predictors in model_families.items():
        for x, z, label in interaction_pairs:
            predictors = list(dict.fromkeys(base_predictors + [x, z]))
            term = f'z_{x}:z_{z}'
            rhs_terms = ['z_' + v for v in predictors] + [term, 'C(iso_code)']
            d = std.dropna(subset=[outcome] + ['z_' + v for v in predictors] + ['iso_code']).copy()
            try:
                fit = fit_fractional_logit(d, outcome, rhs_terms)
                ci = fit.conf_int()
                b=float(fit.params[term]); se=float(fit.bse[term]); p=float(fit.pvalues[term])
                lo, hi = ci.loc[term].tolist()
                factor = ame_factor(fit, d)
                interaction_rows.append({
                    'outcome': outcome, 'model': model_name, 'interaction': label, 'x': x, 'moderator': z,
                    'logit_coef_interaction': b, 'cluster_se': se, 'ci_low': float(lo), 'ci_high': float(hi),
                    'p_value': p, 'stars_raw': star(p),
                    'AME_interaction_approx': b * factor,
                    'AME_ci_low_approx': float(lo) * factor,
                    'AME_ci_high_approx': float(hi) * factor,
                    'ame_factor_mean_mu_1_minus_mu': factor,
                    'n_obs': int(fit.nobs), 'n_countries': d['iso_code'].nunique(),
                    'aic': fit.aic, 'bic_llf': getattr(fit, 'bic_llf', np.nan),
                    'included_predictors': ', '.join(predictors)
                })
                fit_rows.append({'outcome': outcome, 'model': model_name, 'interaction': label, 'status': 'ok'})

                cov = fit.cov_params()
                for focal, mod in [(x, z), (z, x)]:
                    focal_term = 'z_' + focal
                    int_term = term
                    for level_name, mod_value in [('low_-1SD', -1.0), ('mean_0SD', 0.0), ('high_+1SD', 1.0)]:
                        slope = float(fit.params[focal_term] + fit.params[int_term] * mod_value)
                        var_slope = float(cov.loc[focal_term, focal_term] + (mod_value ** 2) * cov.loc[int_term, int_term] + 2 * mod_value * cov.loc[focal_term, int_term])
                        se_slope = np.sqrt(max(var_slope, 0))
                        z_stat = slope / se_slope if se_slope > 0 else np.nan
                        p_slope = 2 * (1 - stats.norm.cdf(abs(z_stat))) if np.isfinite(z_stat) else np.nan

                        d_level = d.copy()
                        d_level['z_' + mod] = mod_value
                        mu_level = fit.predict(d_level)
                        factor_level = float(np.mean(mu_level * (1 - mu_level)))
                        simple_slope_rows.append({
                            'outcome': outcome, 'model': model_name, 'interaction': label,
                            'focal_variable': focal, 'moderator_variable': mod,
                            'moderator_level': level_name, 'moderator_z_value': mod_value,
                            'slope_logit_per_sd': slope, 'se_logit': se_slope,
                            'ci_low_logit': slope - 1.96 * se_slope,
                            'ci_high_logit': slope + 1.96 * se_slope,
                            'p_value': p_slope, 'stars': star(p_slope),
                            'AME_per_sd_at_level': slope * factor_level,
                            'ame_factor_at_level': factor_level,
                            'n_obs': int(fit.nobs), 'n_countries': d['iso_code'].nunique()
                        })
            except Exception as e:
                fit_rows.append({'outcome': outcome, 'model': model_name, 'interaction': label, 'status': 'failed', 'error': repr(e)})

interaction_frac = pd.DataFrame(interaction_rows)
if not interaction_frac.empty:
    for (outcome, model_name), idx in interaction_frac.groupby(['outcome', 'model']).groups.items():
        pvals = interaction_frac.loc[idx, 'p_value'].values
        rejected, qvals = fdrcorrection(pvals, alpha=0.05)
        interaction_frac.loc[idx, 'q_value_fdr_within_outcome_model'] = qvals
        interaction_frac.loc[idx, 'fdr_reject_0_05'] = rejected
    interaction_frac['stars_fdr'] = interaction_frac['q_value_fdr_within_outcome_model'].apply(star)

simple_slopes_frac = pd.DataFrame(simple_slope_rows)
fit_status_frac = pd.DataFrame(fit_rows)
interaction_frac.to_csv(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_interaction_terms_M1_M4.csv', index=False)
simple_slopes_frac.to_csv(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_interaction_simple_slopes_AME_M1_M4.csv', index=False)
fit_status_frac.to_csv(PROCESS_DIR / 'rq3_gap_deficit_fractional_logit_interaction_fit_status.csv', index=False)
interaction_frac.sort_values(['outcome','model','q_value_fdr_within_outcome_model']).head(40)


In [ ]:

# Visual summaries for quick model-selection review.
if not region_frac_coef.empty:
    sub = region_frac_coef[region_frac_coef['variable'].isin(['female_share','education'])].copy()
    sub['label'] = sub['variable'].map({'female_share': 'Female share', 'education': 'Education'})
    sub['region'] = pd.Categorical(sub['region'], categories=sorted(sub['region'].dropna().unique()), ordered=True)
    g = sns.FacetGrid(sub, row='outcome', col='model', hue='label', sharex=False, sharey=True, height=3.2, aspect=1.55, margin_titles=True)
    g.map_dataframe(sns.scatterplot, x='AME_per_sd', y='region', s=55)
    for ax in g.axes.flat:
        ax.axvline(0, color='0.35', lw=0.8)
        ax.set_xlabel('AME per 1 SD increase')
        ax.set_ylabel('')
    g.add_legend(title='Variable')
    g.fig.subplots_adjust(top=0.90)
    g.fig.suptitle('Gap/Deficit fractional logit: regional AMEs for female share and education')
    g.savefig(INTERACTION_DIR / 'Figure_RQ3_gap_deficit_region_female_education_AME_M1_M4.png', bbox_inches='tight')
    plt.close(g.fig)

if not interaction_frac.empty:
    plot = interaction_frac.copy()
    plot['sig_raw'] = plot['p_value'].apply(star)
    plot['interaction_label'] = plot['interaction'] + ' ' + plot['sig_raw']
    g = sns.FacetGrid(plot, row='outcome', col='model', sharex=False, sharey=True, height=3.4, aspect=1.45, margin_titles=True)
    g.map_dataframe(sns.scatterplot, x='AME_interaction_approx', y='interaction', s=55, color='#2f6f73')
    for ax in g.axes.flat:
        ax.axvline(0, color='0.35', lw=0.8)
        ax.set_xlabel('Approx. interaction AME')
        ax.set_ylabel('')
    g.fig.subplots_adjust(top=0.90)
    g.fig.suptitle('Gap/Deficit fractional logit: global interaction terms')
    g.savefig(INTERACTION_DIR / 'Figure_RQ3_gap_deficit_interaction_AME_M1_M4.png', bbox_inches='tight')
    plt.close(g.fig)

    heat = interaction_frac.pivot_table(index='interaction', columns=['outcome','model'], values='q_value_fdr_within_outcome_model')
    plt.figure(figsize=(10, max(4, 0.42 * len(heat))))
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis_r', vmin=0, vmax=0.25, cbar_kws={'label': 'FDR q-value'})
    plt.title('Gap/Deficit fractional logit interactions: FDR q-values')
    plt.tight_layout()
    plt.savefig(INTERACTION_DIR / 'Figure_RQ3_gap_deficit_interaction_fdr_heatmap_M1_M4.png', bbox_inches='tight')
    plt.close()

print('Figures saved to:', INTERACTION_DIR)


In [ ]:

# Pull out female and education evidence across these gap/deficit branches.
evidence=[]
if not region_frac_coef.empty:
    sub = region_frac_coef[region_frac_coef['variable'].isin(['education','female_share'])].copy()
    sub['branch'] = 'region_stratified_fractional_logit'
    evidence.append(sub[['branch','outcome','model','region','variable','logit_coef_per_sd','cluster_se','p_value','stars','AME_per_sd','n_obs','n_countries']])
if not interaction_frac.empty:
    sub = interaction_frac[(interaction_frac['x'].isin(['female_share','education']) | interaction_frac['moderator'].isin(['female_share','education']))].copy()
    sub['branch'] = 'global_fractional_logit_interaction'
    sub['region'] = 'Global'
    sub = sub.rename(columns={'logit_coef_interaction':'logit_coef_per_sd', 'AME_interaction_approx':'AME_per_sd'})
    evidence.append(sub[['branch','outcome','model','region','interaction','x','moderator','logit_coef_per_sd','cluster_se','p_value','q_value_fdr_within_outcome_model','stars_raw','stars_fdr','AME_per_sd','n_obs','n_countries']])
if not simple_slopes_frac.empty:
    sub = simple_slopes_frac[simple_slopes_frac['focal_variable'].isin(['female_share','education'])].copy()
    sub['branch'] = 'global_fractional_logit_simple_slope'
    sub['region'] = 'Global'
    sub = sub.rename(columns={'slope_logit_per_sd':'logit_coef_per_sd', 'AME_per_sd_at_level':'AME_per_sd'})
    evidence.append(sub[['branch','outcome','model','region','interaction','focal_variable','moderator_variable','moderator_level','logit_coef_per_sd','se_logit','p_value','stars','AME_per_sd','n_obs','n_countries']])

evidence_gap_deficit = pd.concat(evidence, ignore_index=True, sort=False) if evidence else pd.DataFrame()
evidence_gap_deficit.to_csv(TABLE_DIR / 'rq3_gap_deficit_female_education_evidence_all_branches.csv', index=False)
evidence_gap_deficit.head(80)


In [ ]:

# Compact manifest for reproducibility.
manifest = {
    'notebook': str(PROJECT / 'RQ3_gap_deficit_model_selection.ipynb'),
    'data_file': str(DATA_PATH),
    'results_dir': str(RESULTS_DIR),
    'tables_dir': str(TABLE_DIR),
    'interaction_dir': str(INTERACTION_DIR),
    'process_dir': str(PROCESS_DIR),
    'sample_n': int(len(std)),
    'countries': int(std['iso_code'].nunique()),
    'regions': int(std['analytical_region'].nunique()),
    'outcomes': outcomes,
    'model_families': model_families,
    'interaction_pairs': [{'x': x, 'moderator': z, 'label': label} for x, z, label in interaction_pairs],
    'outputs': {
        'region_coefficients': str(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_region_coefficients_M1_M4.csv'),
        'region_summary': str(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_region_summary_M1_M4.csv'),
        'interaction_terms': str(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_interaction_terms_M1_M4.csv'),
        'interaction_simple_slopes_AME': str(TABLE_DIR / 'rq3_gap_deficit_fractional_logit_interaction_simple_slopes_AME_M1_M4.csv'),
        'female_education_evidence': str(TABLE_DIR / 'rq3_gap_deficit_female_education_evidence_all_branches.csv')
    }
}
(PROCESS_DIR / 'rq3_gap_deficit_model_selection_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
